In [ ]:
!python --version

In [ ]:
!pip install torch==2.10.0 torchvision --index-url https://download.pytorch.org/whl/cu128

In [ ]:
!git clone https://github.com/facebookresearch/sam3.git
%cd sam3
!pip install -e .

In [ ]:
!hf auth login

In [ ]:
from transformers import Sam3VideoModel, Sam3VideoProcessor
import torch

model = Sam3VideoModel.from_pretrained("facebook/sam3", device_map="auto")
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
!pip install av

In [ ]:
from transformers.video_utils import load_video
video_url = "https://huggingface.co/datasets/hf-internal-testing/sam2-fixtures/resolve/main/bedroom.mp4"
video_frames, _ = load_video(video_url)

In [ ]:
inference_session = processor.init_video_session(
    video=video_frames,
    inference_device=device,
    processing_device="cpu",
    video_storage_device="cpu",
)

In [ ]:
text = "person"
inference_session = processor.add_text_prompt(
    inference_session=inference_session,
    text=text,
)

In [ ]:
# Process all frames in the video
outputs_per_frame = {}
# Pass show_progress_bar=True to display a tqdm progress bar.
for model_outputs in model.propagate_in_video_iterator(
    inference_session=inference_session, max_frame_num_to_track=50
):
    processed_outputs = processor.postprocess_outputs(inference_session, model_outputs)
    outputs_per_frame[model_outputs.frame_idx] = processed_outputs

print(f"Processed {len(outputs_per_frame)} frames")
# Processed 51 frames

# Access results for a specific frame
frame_0_outputs = outputs_per_frame[0]
print(f"Detected {len(frame_0_outputs['object_ids'])} objects")
print(f"Object IDs: {frame_0_outputs['object_ids'].tolist()}")
print(f"Scores: {frame_0_outputs['scores'].tolist()}")
print(f"Boxes shape (XYXY format, absolute coordinates): {frame_0_outputs['boxes'].shape}")
print(f"Masks shape: {frame_0_outputs['masks'].shape}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

frame_idx = 10

frame = video_frames[frame_idx]
result = outputs_per_frame[frame_idx]

masks = result["masks"]

plt.figure(figsize=(10, 10))
plt.imshow(frame)

for mask in masks:
    mask = mask.cpu().numpy()

    plt.imshow(
        np.ma.masked_where(mask == 0, mask),
        alpha=0.5
    )

plt.axis("off")
plt.show()

In [ ]:
import cv2
import numpy as np

height, width = video_frames[0].shape[:2]

writer = cv2.VideoWriter(
    "sam3_result.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    30,
    (width, height)
)

for frame_idx in sorted(outputs_per_frame.keys()):

    frame = video_frames[frame_idx].copy()

    result = outputs_per_frame[frame_idx]

    for mask in result["masks"]:
        mask = mask.cpu().numpy().astype(bool)

        frame[mask] = (
            0.5 * frame[mask] +
            0.5 * np.array([255, 0, 0])
        ).astype(np.uint8)

    writer.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))

writer.release()